In [1]:
import numpy as np
import pandas as pd
from paraphrase_detection import data_shuffle_split, PairClassifier, Train, TextPairDataset, log_metrics_and_model,\
log_bo_results, BOSearchTrain
import torch
from sentence_transformers import SentenceTransformer
from transformers import get_linear_schedule_with_warmup
from torch.utils.data import DataLoader
from loguru import logger
import os
from transformers import AutoTokenizer
from enums import Optimizer, Scheduler

os.environ["TOKENIZERS_PARALLELISM"] = "false"

/Users/oli.dmrs/Documents/Personal Projects/paraphrase-detection-ml/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_parquet("hf://datasets/sentence-transformers/quora-duplicates/pair-class/train-00000-of-00001.parquet")
data = df.to_numpy()
data = data[:1000]

X = data[:, :2]
y = data[:, 2:3]

X_train, X_val, X_test, y_train, y_val, y_test = data_shuffle_split(X, y, 0.15, 0.12, 42)

In [3]:
seed = 42
sbert_model = SentenceTransformer("all-MiniLM-L6-v2")
fc_layer_sizes = [sbert_model.get_sentence_embedding_dimension(), sbert_model.get_sentence_embedding_dimension() // 2]
dropout_p = 0.1
lr = 1e-3
epochs = 10
batch_size = 32
steps = epochs * X_train.shape[0] // batch_size
n_warmup_steps = steps * 0.1
n_freeze = 4
weight_decay = 0.01

np.random.seed(seed)
device = (torch.device("mps") if torch.backends.mps.is_available() else torch.device("cuda" if torch.cuda.is_available() else "cpu"))
model = PairClassifier.CrossEntropy(sbert_model, fc_layer_sizes, 2, device, True, dropout_p)
optimizer = torch.optim.AdamW(model.parameters(), lr, weight_decay = weight_decay)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps = n_warmup_steps, num_training_steps = steps)
criterion = torch.nn.CrossEntropyLoss()

In [10]:
train_loader = DataLoader(dataset = TextPairDataset(X_train, y_train), batch_size = batch_size, shuffle = True, num_workers = 4)
validation_loader = DataLoader(dataset = TextPairDataset(X_val, y_val), batch_size = batch_size, shuffle = True, num_workers = 4)
test_loader = DataLoader(dataset = TextPairDataset(X_test, y_test), batch_size = batch_size, shuffle = True, num_workers = 4)

In [11]:
#Trainable S bert
train = Train(model = model,
              optimizer = optimizer,
              scheduler = scheduler,
              criterion = criterion, 
              device = device,
              n_freeze = n_freeze,
              epochs = epochs,
              train_dataloader = train_loader,
              val_dataloader = validation_loader,
              patience = 3
              )

best_params, avg_batch_train_loss, epoch_train_acc, avg_batch_val_loss, epoch_val_acc, epoch_val_f1 = train.run_training_loop()

EPOCH: 0


2025-11-23 14:21:19.347 | INFO     | paraphrase_detection.train:run_training_loop:171 - Epoch 0: train loss = 0.6121411149700483
2025-11-23 14:21:19.348 | INFO     | paraphrase_detection.train:run_training_loop:172 - Epoch 0: validation loss = 0.5881509855389595
2025-11-23 14:21:19.349 | INFO     | paraphrase_detection.train:run_training_loop:173 - Epoch 0: train acc = 0.6403743315508021
2025-11-23 14:21:19.350 | INFO     | paraphrase_detection.train:run_training_loop:174 - Epoch 0: validation acc = 0.7058823529411765
2025-11-23 14:21:19.351 | INFO     | paraphrase_detection.train:run_training_loop:175 - Epoch 0: validation f1 = 0.6128542510121457


EPOCH: 1


2025-11-23 14:22:06.133 | INFO     | paraphrase_detection.train:run_training_loop:171 - Epoch 1: train loss = 0.3362288276354472
2025-11-23 14:22:06.135 | INFO     | paraphrase_detection.train:run_training_loop:172 - Epoch 1: validation loss = 0.41300116665661335
2025-11-23 14:22:06.136 | INFO     | paraphrase_detection.train:run_training_loop:173 - Epoch 1: train acc = 0.8516042780748663
2025-11-23 14:22:06.137 | INFO     | paraphrase_detection.train:run_training_loop:174 - Epoch 1: validation acc = 0.7843137254901961
2025-11-23 14:22:06.138 | INFO     | paraphrase_detection.train:run_training_loop:175 - Epoch 1: validation f1 = 0.7801724137931034


EPOCH: 2


2025-11-23 14:22:52.994 | INFO     | paraphrase_detection.train:run_training_loop:171 - Epoch 2: train loss = 0.1711206097776691
2025-11-23 14:22:52.996 | INFO     | paraphrase_detection.train:run_training_loop:172 - Epoch 2: validation loss = 0.44119973480701447
2025-11-23 14:22:52.997 | INFO     | paraphrase_detection.train:run_training_loop:173 - Epoch 2: train acc = 0.9344919786096256
2025-11-23 14:22:52.998 | INFO     | paraphrase_detection.train:run_training_loop:174 - Epoch 2: validation acc = 0.7745098039215687
2025-11-23 14:22:52.998 | INFO     | paraphrase_detection.train:run_training_loop:175 - Epoch 2: validation f1 = 0.7680672268907562


EPOCH: 3


2025-11-23 14:23:39.699 | INFO     | paraphrase_detection.train:run_training_loop:171 - Epoch 3: train loss = 0.07388817084332307
2025-11-23 14:23:39.700 | INFO     | paraphrase_detection.train:run_training_loop:172 - Epoch 3: validation loss = 0.9367796033620834
2025-11-23 14:23:39.701 | INFO     | paraphrase_detection.train:run_training_loop:173 - Epoch 3: train acc = 0.9852941176470589
2025-11-23 14:23:39.701 | INFO     | paraphrase_detection.train:run_training_loop:174 - Epoch 3: validation acc = 0.7450980392156863
2025-11-23 14:23:39.702 | INFO     | paraphrase_detection.train:run_training_loop:175 - Epoch 3: validation f1 = 0.7349060375849661


EPOCH: 4


2025-11-23 14:24:26.702 | INFO     | paraphrase_detection.train:run_training_loop:171 - Epoch 4: train loss = 0.02247348064944769
2025-11-23 14:24:26.704 | INFO     | paraphrase_detection.train:run_training_loop:172 - Epoch 4: validation loss = 0.8409128412604332
2025-11-23 14:24:26.705 | INFO     | paraphrase_detection.train:run_training_loop:173 - Epoch 4: train acc = 0.9973262032085561
2025-11-23 14:24:26.705 | INFO     | paraphrase_detection.train:run_training_loop:174 - Epoch 4: validation acc = 0.7745098039215687
2025-11-23 14:24:26.706 | INFO     | paraphrase_detection.train:run_training_loop:175 - Epoch 4: validation f1 = 0.7664044608184806
2025-11-23 14:24:26.709 | WARNING  | paraphrase_detection.train:run_training_loop:186 - EARLY STOP during training at epoch 4


### Bayesian Optimization search for CrossEntropy model

In [ ]:
# sbert_model = SentenceTransformer("all-MiniLM-L12-v2")

# search_params = ['lr', 'weight_decay', 'dropout_p', 'batch_size', 'n_freeze', 'use_n_layers', 'fc1', 'fc2', 'fc3']

# bounds = torch.tensor([
#     [1e-5, 0.001, 0.0, 16.0, 9.0, 1.0, 64.0, 64.0, 16.0],
#     [1e-1, 0.1, 0.6, 512.0, sbert_model[0].auto_model.config.num_hidden_layers, 3.0, 256.0, 128.0, 64.0]
# ])

# seed = 42
# criterion = torch.nn.CrossEntropyLoss()
# device = (torch.device("mps") if torch.backends.mps.is_available() else torch.device("cuda" if torch.cuda.is_available() else "cpu"))

# bo_searcher = BOSearchTrain(
#     bounds = bounds,
#     names = search_params,
#     sbert_model = sbert_model,
#     X_train = X_train,
#     y_train = y_train,
#     X_val = X_val,
#     y_val = y_val,
#     optimizer = Optimizer.ADAMW,
#     scheduler = Scheduler.LINEAR,
#     criterion = criterion,
#     device = device,
#     n_init_samples = 10,
#     n_iterations = 30,
#     fixed = False,
#     cos_similarity = False,
#     epochs = 10,
#     patience = 3
# )

# best_results = bo_searcher.search_loop()

2025-11-23 13:14:29.794 | INFO     | paraphrase_detection.hyperparameterselection:search_loop:345 - Starting BO hyperparameter search 

2025-11-23 13:14:29.795 | INFO     | paraphrase_detection.hyperparameterselection:init_fit_samples:158 - Initial fit of GP for 10 samples of hyperparameter sets 

2025-11-23 13:14:29.796 | INFO     | paraphrase_detection.hyperparameterselection:init_fit_samples:167 - Starting iteration 0:
[('lr', 0.08401218056678772), ('weight_decay', 0.07427217811346054), ('dropout', 0.09737738966941833), ('batch_size', 165), ('n_freeze', 10), ('use_n_layers', 2), ('fc1', 221), ('fc2', 126), ('fc3', 20)]


EPOCH: 0


### TEST

In [3]:
sbert_model = SentenceTransformer("paraphrase-MiniLM-L6-v2")

search_params = ['lr', 'weight_decay', 'dropout_p', 'batch_size', 'n_freeze', 'use_n_layers', 'fc1', 'fc2', 'fc3']

bounds = torch.tensor([
    [1e-4, 0.001, 0.0, 16.0, 3.0, 1.0, 64.0, 64.0, 16.0], #min 
    [3e-3, 0.1, 0.5, 96.0, sbert_model[0].auto_model.config.num_hidden_layers, 3.0, 256.0, 128.0, 64.0] #max
])

seed = 41

np.random.seed(seed)
torch.manual_seed(seed)

criterion = torch.nn.CrossEntropyLoss()
device = (torch.device("mps") if torch.backends.mps.is_available() else torch.device("cuda" if torch.cuda.is_available() else "cpu"))

bo_searcher = BOSearchTrain(
    bounds = bounds,
    names = search_params,
    sbert_model = sbert_model,
    X_train = X_train,
    y_train = y_train,
    X_val = X_val,
    y_val = y_val,
    optimizer = Optimizer.ADAMW,
    scheduler = Scheduler.LINEAR,
    criterion = criterion,
    device = device,
    n_init_samples = 2,
    n_iterations = 4,
    fixed = True,
    cos_similarity = False,
    epochs = 1,
    patience = 3
)

best_results = bo_searcher.search_loop()

2025-11-23 17:10:35.133 | INFO     | paraphrase_detection.hyperparameterselection:init_fit_samples:158 - Initial fit of GP for 2 samples of hyperparameter sets 

2025-11-23 17:10:35.134 | INFO     | paraphrase_detection.hyperparameterselection:init_fit_samples:167 - Starting iteration 0:
[('lr', 0.0007856945157982409), ('weight_decay', 0.023435181006789207), ('dropout', 0.400265097618103), ('batch_size', 29), ('n_freeze', 3), ('use_n_layers', 2), ('fc1', 88), ('fc2', 111), ('fc3', 54)]


EPOCH: 0


2025-11-23 17:11:22.515 | INFO     | paraphrase_detection.train:run_training_loop:170 - Epoch 0: train loss = 0.6182448806670996
2025-11-23 17:11:22.516 | INFO     | paraphrase_detection.train:run_training_loop:171 - Epoch 0: validation loss = 0.5196656882762909
2025-11-23 17:11:22.517 | INFO     | paraphrase_detection.train:run_training_loop:172 - Epoch 0: train acc = 0.660427807486631
2025-11-23 17:11:22.517 | INFO     | paraphrase_detection.train:run_training_loop:173 - Epoch 0: validation acc = 0.6862745098039216
2025-11-23 17:11:22.518 | INFO     | paraphrase_detection.train:run_training_loop:174 - Epoch 0: validation f1 = 0.597037037037037
2025-11-23 17:11:22.522 | INFO     | paraphrase_detection.hyperparameterselection:init_fit_samples:176 - Maximum validation F1 score: 0.597037037037037 

2025-11-23 17:11:22.523 | INFO     | paraphrase_detection.hyperparameterselection:init_fit_samples:167 - Starting iteration 1:
[('lr', 0.001943556359037757), ('weight_decay', 0.059372022747993

EPOCH: 0


2025-11-23 17:12:00.968 | INFO     | paraphrase_detection.train:run_training_loop:170 - Epoch 0: train loss = 0.7062633302476671
2025-11-23 17:12:00.969 | INFO     | paraphrase_detection.train:run_training_loop:171 - Epoch 0: validation loss = 0.5551319718360901
2025-11-23 17:12:00.970 | INFO     | paraphrase_detection.train:run_training_loop:172 - Epoch 0: train acc = 0.6443850267379679
2025-11-23 17:12:00.970 | INFO     | paraphrase_detection.train:run_training_loop:173 - Epoch 0: validation acc = 0.7450980392156863
2025-11-23 17:12:00.971 | INFO     | paraphrase_detection.train:run_training_loop:174 - Epoch 0: validation f1 = 0.7326612903225806
2025-11-23 17:12:00.976 | INFO     | paraphrase_detection.hyperparameterselection:init_fit_samples:176 - Maximum validation F1 score: 0.7326612903225806 

2025-11-23 17:12:00.977 | INFO     | paraphrase_detection.hyperparameterselection:init_fit_samples:177 - Finished initial fit of samples
2025-11-23 17:12:01.031 | INFO     | paraphrase_dete

...

2025-11-23 17:13:16.728 | INFO     | paraphrase_detection.train:run_training_loop:170 - Epoch 0: train loss = 0.7587223052978516
2025-11-23 17:13:16.730 | INFO     | paraphrase_detection.train:run_training_loop:171 - Epoch 0: validation loss = 0.4928000122308731
2025-11-23 17:13:16.731 | INFO     | paraphrase_detection.train:run_training_loop:172 - Epoch 0: train acc = 0.6577540106951871
2025-11-23 17:13:16.731 | INFO     | paraphrase_detection.train:run_training_loop:173 - Epoch 0: validation acc = 0.7352941176470589
2025-11-23 17:13:16.732 | INFO     | paraphrase_detection.train:run_training_loop:174 - Epoch 0: validation f1 = 0.7321793250996791
2025-11-23 17:13:16.737 | INFO     | paraphrase_detection.hyperparameterselection:search_loop:380 - 
 Maximum validation F1 score: 0.7321793250996791 

2025-11-23 17:13:16.844 | INFO     | paraphrase_detection.hyperparameterselection:search_loop:365 - Starting iteration 2:
[('lr', 0.0029999999605934136), ('weight_decay', 0.07652935421459676),

EPOCH: 0


2025-11-23 17:13:53.401 | INFO     | paraphrase_detection.train:run_training_loop:170 - Epoch 0: train loss = 0.6709685325622559
2025-11-23 17:13:53.402 | INFO     | paraphrase_detection.train:run_training_loop:171 - Epoch 0: validation loss = 0.6207014173269272
2025-11-23 17:13:53.403 | INFO     | paraphrase_detection.train:run_training_loop:172 - Epoch 0: train acc = 0.68048128342246
2025-11-23 17:13:53.403 | INFO     | paraphrase_detection.train:run_training_loop:173 - Epoch 0: validation acc = 0.7843137254901961
2025-11-23 17:13:53.403 | INFO     | paraphrase_detection.train:run_training_loop:174 - Epoch 0: validation f1 = 0.7638888888888888
2025-11-23 17:13:53.407 | INFO     | paraphrase_detection.hyperparameterselection:search_loop:380 - 
 Maximum validation F1 score: 0.7638888888888888 

2025-11-23 17:13:53.489 | INFO     | paraphrase_detection.hyperparameterselection:search_loop:365 - Starting iteration 3:
[('lr', 0.0029999999605934136), ('weight_decay', 0.09999999951105565), (

EPOCH: 0


2025-11-23 17:14:30.630 | INFO     | paraphrase_detection.train:run_training_loop:170 - Epoch 0: train loss = 0.8192988559603691
2025-11-23 17:14:30.632 | INFO     | paraphrase_detection.train:run_training_loop:171 - Epoch 0: validation loss = 0.5946839153766632
2025-11-23 17:14:30.633 | INFO     | paraphrase_detection.train:run_training_loop:172 - Epoch 0: train acc = 0.6136363636363636
2025-11-23 17:14:30.634 | INFO     | paraphrase_detection.train:run_training_loop:173 - Epoch 0: validation acc = 0.7647058823529411
2025-11-23 17:14:30.635 | INFO     | paraphrase_detection.train:run_training_loop:174 - Epoch 0: validation f1 = 0.7455301455301455
2025-11-23 17:14:30.641 | INFO     | paraphrase_detection.hyperparameterselection:search_loop:380 - 
 Maximum validation F1 score: 0.7455301455301455 



In [5]:
# Saves best model, HP samples with F1 scores and metrics of best performing model
log_bo_results(
    model_name = f'BO_cos{bo_searcher.cos_similarity}',
    X_observed = bo_searcher.X_observed.numpy(),
    Y_observed = bo_searcher.Y_observed.numpy(),
    col_names = bo_searcher.names,
    all_metrics = best_results[1:6],
    params = best_results[0]
)

ValueError: all the input arrays must have same number of dimensions, but the array at index 0 has 1 dimension(s) and the array at index 1 has 2 dimension(s)

In [3]:
seed = 42
sbert_model = SentenceTransformer("all-MiniLM-L6-v2")
fc_layer_sizes = [sbert_model.get_sentence_embedding_dimension(), sbert_model.get_sentence_embedding_dimension() // 2]
dropout_p = 0.1
lr = 1e-3
epochs = 10
batch_size = 32
steps = epochs * X_train.shape[0] // batch_size
n_warmup_steps = steps * 0.1
n_freeze = 3
weight_decay = 0.01

np.random.seed(seed)
device = (torch.device("mps") if torch.backends.mps.is_available() else torch.device("cuda" if torch.cuda.is_available() else "cpu"))
model = SBERTPairClassifier(sbert_model, fc_layer_sizes, device, True, dropout_p)
optimizer = torch.optim.AdamW(model.parameters(), lr, weight_decay = weight_decay)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps = n_warmup_steps, num_training_steps = steps)
criterion = torch.nn.CrossEntropyLoss()

ModuleList(
  (0): Linear(in_features=1152, out_features=384, bias=True)
  (1): Linear(in_features=384, out_features=192, bias=True)
  (2): Linear(in_features=192, out_features=2, bias=True)
)


In [ ]:
train_loader = DataLoader(dataset = TextPairDataset(X_train, y_train), batch_size = batch_size, shuffle = True, num_workers = 4)
validation_loader = DataLoader(dataset = TextPairDataset(X_val, y_val), batch_size = batch_size, shuffle = True, num_workers = 4)
test_loader = DataLoader(dataset = TextPairDataset(X_test, y_test), batch_size = batch_size, shuffle = True, num_workers = 4)

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
tokenized_train_loader = DataLoader(dataset = TextPairDataset(X_train, y_train, tokenization=True, tokenizer=tokenizer), batch_size = batch_size, shuffle = True, num_workers = 4)
tokenized_validation_loader = DataLoader(dataset = TextPairDataset(X_val, y_val, tokenization=True, tokenizer=tokenizer), batch_size = batch_size, shuffle = True, num_workers = 4)
tokenized_test_loader = DataLoader(dataset = TextPairDataset(X_test, y_test, tokenization=True, tokenizer=tokenizer), batch_size = batch_size, shuffle = True, num_workers = 4)

In [ ]:
#Fixed sbert model
train = Train(model = model,
              optimizer = optimizer,
              scheduler = scheduler,
              criterion = criterion, 
              device = device,
              n_freeze = n_freeze,
              epochs = epochs,
              train_dataloader = tokenized_train_loader,
              val_dataloader = tokenized_test_loader,
              sbert_trainable = True
              )

best_params, avg_batch_train_loss, epoch_train_acc, avg_batch_val_loss, epoch_val_acc, epoch_val_f1 = train.run_training_loop()

False
False
False
False
False
EPOCH: 0


2025-11-20 18:34:35.229 | INFO     | paraphrase_detection.train:run_training_loop:126 - Epoch 0: train loss = 0.5515606858027287
2025-11-20 18:34:35.231 | INFO     | paraphrase_detection.train:run_training_loop:127 - Epoch 0: validation loss = 0.6087496508943274
2025-11-20 18:34:35.231 | INFO     | paraphrase_detection.train:run_training_loop:128 - Epoch 0: train acc = 0.707620320855615
2025-11-20 18:34:35.232 | INFO     | paraphrase_detection.train:run_training_loop:129 - Epoch 0: validation acc = 0.642
2025-11-20 18:34:35.232 | INFO     | paraphrase_detection.train:run_training_loop:130 - Epoch 0: validation f1 = 0.6419807465201461


EPOCH: 1


In [6]:
log_metrics_and_model('SBERTv1', avg_batch_train_loss, epoch_train_acc, avg_batch_val_loss, \
                      epoch_val_acc, epoch_val_f1, best_params)